# Comparação Metrológica e Calibração de Células de Carga (GUM 2008 - Anexo H.3)

 Este notebook Jupyter foi desenvolvido para realizar a calibração de múltiplos pontos de uma célula de carga de 1 kg de capacidade, comparando dois sistemas de aquisição:

1. **Quantum MX1615 (HBM)** - Sistema profissional de alta exatidão.
2. **Arduino + HX711** - Sistema de baixo custo de desenvolvimento próprio.


A análise de incertezas e o ajuste de curva de calibração seguem estritamente as diretrizes do **JCGM GUM 2008**.

In [ ]:
import pandas as pd #Funciona como uma planilha do Excel
import numpy as np #Realiza operações matemáticas complexas, álgebra e arrays
import matplotlib.pyplot as plt #Criação e customização de gráficos
import scipy.stats as stats #Estatística avançada (Ciência e Engenharia)
import os

print("Bibliotecas importadas com sucesso!")

## 1. Carregamento dos Dados

Os dados experimentais de 10 medições repetidas para os pesos de 50g, 100g, 200g e 500g são carregados de arquivos CSV.

In [ ]:
# Dados de exemplo simulados de acordo com os arquivos csv disponibilizados
def create_default_csv_if_missing():
    if not os.path.exists('dados_sistema_a.csv'): #Caso não possua o arquivo
        data_ard = {
            'peso_padrao': [50]*10 + [100]*10 + [200]*10 + [500]*10,
            'leitura': (
                [50.1, 50.0, 50.1, 50.0, 50.1, 50.0, 50.1, 50.1, 50.0, 50.0] +
                [100.1, 100.0, 100.1, 100.0, 100.1, 100.0, 100.1, 100.0, 100.1, 100.1] +
                [200.1, 200.0, 200.1, 200.0, 200.1, 200.1, 200.0, 200.0, 200.1, 200.1] +
                [500.1, 500.0, 500.1, 500.0, 500.1, 500.1, 500.0, 500.0, 500.1, 500.1]
            )
        }
        pd.DataFrame(data_ard).to_csv('dados_sistema_a.csv', index=False) #Gera um novo arquivo com esses dados

    if not os.path.exists('dados_sistema_b.csv'): #Caso não possua o arquivo
        data_quant = {
            'peso_padrao': [50]*10 + [100]*10 + [200]*10 + [500]*10,
            'leitura': (
                [50.002, 50.001, 50.003, 50.001, 50.002, 50.000, 50.002, 50.001, 50.002, 50.001] +
                [100.004, 100.003, 100.005, 100.003, 100.004, 100.002, 100.004, 100.003, 100.004, 100.004] +
                [200.006, 200.005, 200.007, 200.005, 200.006, 200.004, 200.006, 200.005, 200.006, 200.006] +
                [500.012, 500.010, 500.015, 500.011, 500.013, 500.009, 500.014, 500.011, 500.012, 500.013]
            )
        }
        pd.DataFrame(data_quant).to_csv('dados_sistema_b.csv', index=False) #Gera um novo arquivo com esses dados

create_default_csv_if_missing()
df_sistema_a = pd.read_csv('dados_sistema_a.csv')
df_sistema_b = pd.read_csv('dados_sistema_b.csv')
print(df_sistema_b.head(15))
df_sistema_b.head(15)
print("Arquivos carregados com sucesso!")
print("Tamanho Arduino:", df_sistema_a.shape, "| Tamanho Quantum:", df_sistema_b.shape)

## 2. Processamento Estatístico e Orçamento de Incerteza (GUM)

Para cada peso nominal, calcula-se a média, desvio padrão amostral, incerteza de repetibilidade (Tipo A), incerteza do peso de calibração (certificado), resolução do display e exatidão do amplificador (Tipo B).

In [ ]:
# Parâmetros metrológicos
u_cal = 0.0006  # Incerteza padrão do peso padrão (g) obtido via certificado
res_ard = 0.1   # Resolução Arduino (g)
res_quant = 0.001 # Resolução Quantum (g)

u_res_ard = res_ard / np.sqrt(12)
u_res_quant = res_quant / np.sqrt(12)

print("Componentes de resolução:")
print(f"Sistema A: δ = {res_ard} g | u_res = {u_res_ard:.6f} g")
print(f"Sistema B: δ = {res_quant} g | u_res = {u_res_quant:.6f} g\n")

# Incertezas do Amplificador (Tipo B)
# Arduino (HX711): não-linearidade max de 0.02% do fundo de escala (1000g) = 0.2g
u_amp_ard = 0.2 / np.sqrt(3)
# Quantum: exatidão de 0.05% da leitura (dinâmico por ponto)
def get_u_amp_quant(leitura_media):
    return (leitura_media * 0.0005) / np.sqrt(3)

def calcular_tabela_incertezas(df, u_res, is_arduino=True):
    pontos = df['peso_padrao'].unique()
    resultados = []

    for p in pontos:
        sub_df = df[df['peso_padrao'] == p]
        leituras = sub_df['leitura'].values

        media = np.mean(leituras)
        desvio = np.std(leituras, ddof=1)
        u_A = desvio / np.sqrt(len(leituras)) #Repetibilidade

        if is_arduino:
            u_amp = u_amp_ard
        else:
            u_amp = get_u_amp_quant(media)

        # Incerteza padrão combinada
        u_c = np.sqrt(u_A**2 + u_cal**2 + u_res**2 + u_amp**2)

        # Welch-Satterthwaite para graus de liberdade efetivos
        v_eff = 9 * (u_c / u_A)**4 if u_A > 0 else float('inf')

        # Fator de abrangência k para 95% de confiança (t de Student)
        k = stats.t.ppf(0.975, df=v_eff) if v_eff < 1000 else 1.960
        if np.isnan(k) or np.isinf(k):
            k = 1.960

        U = k * u_c

        resultados.append({
            'peso_padrao': p,
            'media': media,
            'u_A': u_A,
            'u_cal': u_cal,
            'u_res': u_res,
            'u_amp': u_amp,
            'u_c': u_c,
            'v_eff': v_eff,
            'k': k,
            'U': U
        })
    return pd.DataFrame(resultados)

res_ard_df = calcular_tabela_incertezas(df_sistema_a, u_res_ard, is_arduino=True)
res_quant_df = calcular_tabela_incertezas(df_sistema_b, u_res_quant, is_arduino=False)

print("=== Tabela Arduino ===")
print(res_ard_df[['peso_padrao', 'media', 'u_A', 'u_c', 'U']])
print("\n=== Tabela Quantum ===")
print(res_quant_df[['peso_padrao', 'media', 'u_A', 'u_c', 'U']])

## 3. Calibração por Mínimos Quadrados (GUM - Anexo H.3)

Ajustamos uma reta de calibração para as correções:
$$ b(t) = y_1 + y_2(t - m_0) $$
com $m_0 = 200\text{ g}$ como referência central.

In [ ]:
def realizar_ajuste_gum(res_df):
    m_0 = 200.0
    t = res_df['media'].values
    b = res_df['peso_padrao'].values - t # Correção b = peso_padrao - leitura
    n = len(t)

    theta = t - m_0
    theta_bar = np.mean(theta)
    b_bar = np.mean(b)

    # Mínimos Quadrados
    num_y2 = np.sum((theta - theta_bar) * (b - b_bar))
    den_y2 = np.sum((theta - theta_bar)**2)
    y2 = num_y2 / den_y2
    y1 = b_bar - y2 * theta_bar

    # Resíduos e variância de ajuste
    b_pred = y1 + y2 * theta
    s_fit = np.sqrt(np.sum((b - b_pred)**2) / (n - 2))

    D = n * np.sum(theta**2) - (np.sum(theta))**2
    u_y1 = s_fit * np.sqrt(np.sum(theta**2) / D)
    u_y2 = s_fit * np.sqrt(n / D)
    cov_y1_y2 = -s_fit**2 * np.sum(theta) / D
    r_y1_y2 = cov_y1_y2 / (u_y1 * u_y2)

    return y1, y2, u_y1, u_y2, r_y1_y2, s_fit, m_0

ajust_ard = realizar_ajuste_gum(res_ard_df)
ajust_quant = realizar_ajuste_gum(res_quant_df)

print("=== Ajuste GUM Arduino ===")
print(f"y1 (Intercepto): {ajust_ard[0]:.6f} g | u(y1): {ajust_ard[2]:.6f} g")
print(f"y2 (Inclinação): {ajust_ard[1]:.6f} | u(y2): {ajust_ard[3]:.6f}")
print(f"r(y1, y2): {ajust_ard[4]:.4f} | Desvio de Ajuste (s_fit): {ajust_ard[5]:.6f} g")

print("\n=== Ajuste GUM Quantum ===")
print(f"y1 (Intercepto): {ajust_quant[0]:.6f} g | u(y1): {ajust_quant[2]:.6f} g")
print(f"y2 (Inclinação): {ajust_quant[1]:.6f} | u(y2): {ajust_quant[3]:.6f}")
print(f"r(y1, y2): {ajust_quant[4]:.4f} | Desvio de Ajuste (s_fit): {ajust_quant[5]:.6f} g")

## 4. Plotagem Gráfica com Bandas de Incerteza de 95%

Visualização das retas de calibração associadas às bandas hiperbólicas de incerteza expandida propagadas.

In [ ]:
def plotar_curva_calibracao(res_df, ajust, titulo, color, filename):
    y1, y2, u_y1, u_y2, r_y1_y2, s_fit, m_0 = ajust
    t_plot = np.linspace(0, 600, 100)
    theta_plot = t_plot - m_0

    b_plot = y1 + y2 * theta_plot

    # Propagação de incerteza da curva conforme GUM H.3
    u_b = np.sqrt(u_y1**2 + (theta_plot**2)*(u_y2**2) + 2*theta_plot*u_y1*u_y2*r_y1_y2)
    U_b = 2.0 * u_b  # k=2 para ~95%

    t_exp = res_df['media'].values
    b_exp = res_df['peso_padrao'].values - t_exp

    plt.figure(figsize=(10, 6))
    plt.plot(t_plot, b_plot, label='Correção Ajustada $b(t)$', color=color, linewidth=2)
    plt.fill_between(t_plot, b_plot - U_b, b_plot + U_b, color=color, alpha=0.15, label='Banda de Incerteza Expandida (95%)')
    plt.scatter(t_exp, b_exp, color='black', zorder=5, label='Dados Experimentais (Médias)')

    plt.title(titulo, fontsize=12, fontweight='bold')
    plt.xlabel('Leitura Indicada $t$ (g)', fontsize=10)
    plt.ylabel('Correção de Calibração $b$ (g)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

plotar_curva_calibracao(res_ard_df, ajust_ard, 'Curva de Calibração GUM - Sistema A', '#E74C3C', 'calibracao_arduino.png')
plotar_curva_calibracao(res_quant_df, ajust_quant, 'Curva de Calibração GUM - Sistema B', '#27AE60', 'calibracao_quantum.png')

## 5. Teste de Compatibilidade Metrológica (Erro Normalizado)

Comparamos a leitura final do ponto de calibração de $500\text{ g}$ de ambos os sistemas para verificar sua equivalência estatística.

In [ ]:
# Comparação para o ponto de 500g
pt_ard = res_ard_df[res_ard_df['peso_padrao'] == 200].iloc[0]
pt_quant = res_quant_df[res_quant_df['peso_padrao'] == 200].iloc[0]

diff = np.abs(pt_ard['media'] - pt_quant['media'])
unc_erro = np.sqrt(pt_ard['U']**2 + pt_quant['U']**2)
En = diff / unc_erro

print("=== COMPATIBILIDADE METROLÓGICA (200g) ===")
print(f"Média Arduino: {pt_ard['media']:.4f} g | Incerteza U_ard: {pt_ard['U']:.4f} g")
print(f"Média Quantum: {pt_quant['media']:.4f} g | Incerteza U_quant: {pt_quant['U']:.4f} g")
print(f"Diferença Absoluta: {diff:.4f} g")
print(f"Incerteza Combinada das Diferenças: {unc_erro:.4f} g")
print(f"Erro Normalizado (En): {En:.3f}")

if En <= 1.0:
    print("\nResultado: COMPATÍVEIS (En <= 1)")
    print("Os sistemas são estatisticamente equivalentes no nível de 95% de confiança.")
else:
    print("\nResultado: DISCREPANTES (En > 1)")
    print("Verificar possíveis erros sistemáticos ou recalibrar os sensores.")

## 6. Avaliação do ajuste por lei de potência

Para verificar a possibilidade de representar a relação entre a massa padrão e a resposta obtida pelos sistemas de aquisição por meio de um modelo não linear, foi realizado um ajuste segundo uma lei de potência, na forma:

$$
y = a x^b
$$

em que $y$ representa a leitura obtida pelo sistema, $x$ corresponde à massa padrão aplicada, e $a$ e $b$ são os parâmetros determinados pelo ajuste.

O ajuste por lei de potência foi comparado ao modelo linear utilizado na calibração, considerando o coeficiente de determinação ($R^2$) como critério de comparação. Para cada massa padrão, foi utilizada a média das dez medições experimentais realizadas.

A análise foi realizada separadamente para os sistemas Sistema A e Sistema B, utilizando os dados experimentais registrados nos respectivos arquivos CSV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score


# ------------------------------------------------------------
# Modelo de potência
# ------------------------------------------------------------

def modelo_potencia(F, a, b):
    return a * F**b


# ------------------------------------------------------------
# Função para analisar uma curva de calibração
# ------------------------------------------------------------

def analisar_curva(dados, nome_curva):

    # --------------------------------------------------------
    # Ajuste linear
    # --------------------------------------------------------

    inclinacao, deslocamento, correlacao, _, _ = stats.linregress(
        dados['forca_N'],
        dados['leitura_media_raw']
    )

    r2_linear = correlacao**2


    # --------------------------------------------------------
    # Ajuste por lei de potência
    # --------------------------------------------------------

    # O ponto de força zero permanece na calibração,
    # mas é excluído somente do ajuste por potência.

    dados_potencia = dados[dados['forca_N'] > 0]

    F = dados_potencia['forca_N'].values
    RAW = dados_potencia['leitura_media_raw'].values

    parametros_potencia, _ = curve_fit(
        modelo_potencia,
        F,
        RAW,
        p0=[1, 1]
    )

    a_pot, b_pot = parametros_potencia


    # Valores previstos pelo modelo de potência

    RAW_potencia = modelo_potencia(
        F,
        a_pot,
        b_pot
    )


    # Coeficiente de determinação do ajuste por potência

    r2_potencia = r2_score(
        RAW,
        RAW_potencia
    )


    # --------------------------------------------------------
    # Resultados
    # --------------------------------------------------------

    print("=" * 70)
    print(nome_curva)
    print("=" * 70)

    print("\nAjuste linear:")
    print(f"Inclinação = {inclinacao:.8f}")
    print(f"Deslocamento = {deslocamento:.8f}")
    print(f"R² linear = {r2_linear:.8f}")

    print("\nAjuste por lei de potência:")
    print(f"RAW = {a_pot:.8f} F^{b_pot:.8f}")
    print(f"Expoente b = {b_pot:.8f}")
    print(f"R² potência = {r2_potencia:.8f}")


    # --------------------------------------------------------
    # Retorna os resultados
    # --------------------------------------------------------

    return {
        'nome': nome_curva,
        'inclinacao': inclinacao,
        'deslocamento': deslocamento,
        'r2_linear': r2_linear,
        'a_potencia': a_pot,
        'b_potencia': b_pot,
        'r2_potencia': r2_potencia
    }


# ============================================================
# FIGURA 1
# ============================================================

df_1 = pd.read_csv(
    'calibracao_a.csv'
)

# Renomeia as colunas para o padrão usado na análise

df_1 = df_1.rename(columns={
    'Massa (g)': 'massa_g',
    'Força (N)': 'forca_N',
    'Leitura RAW': 'leitura_media_raw'
})

dados_1 = df_1[
    ['massa_g', 'forca_N', 'leitura_media_raw']
].copy()


# ============================================================
# FIGURA 2
# ============================================================

df_2 = pd.read_csv(
    'calibracao_b.csv'
)

dados_2 = df_2[
    ['massa_g', 'forca_N', 'leitura_media_raw']
].copy()


# ============================================================
# FIGURA 3
# ============================================================

df_3 = pd.read_csv(
    'calibracao_c.csv'
)

dados_3 = df_3[
    ['massa_g', 'forca_N', 'leitura_media_raw']
].copy()


# ============================================================
# ANÁLISE DAS TRÊS CURVAS
# ============================================================

resultado_1 = analisar_curva(
    dados_1,
    'Figura 1'
)

resultado_2 = analisar_curva(
    dados_2,
    'Figura 2'
)

resultado_3 = analisar_curva(
    dados_3,
    'Figura 3'
)


# ============================================================
# TABELA COMPARATIVA
# ============================================================

resultados = pd.DataFrame([
    resultado_a,
    resultado_b,
    resultado_c
])

print("\n")
print("=" * 70)
print("COMPARAÇÃO DOS AJUSTES")
print("=" * 70)

print(
    resultados[
        [
            'nome',
            'r2_linear',
            'r2_potencia',
            'b_potencia'
        ]
    ].to_string(index=False)
)


# ============================================================
# GRÁFICOS COMPARATIVOS
# ============================================================

curvas = [
    (dados_a, resultado_a, 'Comparação dos ajustes linear e por lei de potência'),
    (dados_b, resultado_b, 'Comparação dos ajustes linear e por lei de potência — curva de calibração de 0 a 350 g'),
    (dados_c, resultado_c, 'Comparação dos ajustes linear e por lei de potência — curva de calibração reprocessada')
]


for dados, resultado, nome in curvas:

    plt.figure(figsize=(10, 6))

    # --------------------------------------------------------
    # Dados experimentais
    # --------------------------------------------------------

    plt.scatter(
        dados['forca_N'],
        dados['leitura_media_raw'],
        label='Dados experimentais'
    )


    # --------------------------------------------------------
    # Faixa de força para desenhar os ajustes
    # --------------------------------------------------------

    forca_plot = np.linspace(
        0,
        dados['forca_N'].max(),
        300
    )


    # --------------------------------------------------------
    # Ajuste linear
    # --------------------------------------------------------

    raw_linear_plot = (
        resultado['inclinacao'] * forca_plot
        + resultado['deslocamento']
    )


    # --------------------------------------------------------
    # Ajuste por potência
    # --------------------------------------------------------

    raw_potencia_plot = modelo_potencia(
        forca_plot,
        resultado['a_potencia'],
        resultado['b_potencia']
    )


    # --------------------------------------------------------
    # Curva linear
    # --------------------------------------------------------

    plt.plot(
        forca_plot,
        raw_linear_plot,
        '--',
        label=(
            f"Ajuste linear "
            f"($R^2$ = {resultado['r2_linear']:.8f})"
        )
    )


    # --------------------------------------------------------
    # Curva de potência
    # --------------------------------------------------------

    plt.plot(
        forca_plot,
        raw_potencia_plot,
        '-.',
        label=(
            f"Lei de potência "
            f"($R^2$ = {resultado['r2_potencia']:.8f})"
        )
    )


    # --------------------------------------------------------
    # Configuração do gráfico
    # --------------------------------------------------------

    plt.xlabel('Força aplicada (N)')
    plt.ylabel('Leitura bruta (RAW)')
    plt.title(nome)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.show()